<a href="https://colab.research.google.com/github/iDurugkar/practice-2026/blob/main/TorchCode/35_bpe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/35_bpe.ipynb)

# 🔴 Hard: Byte-Pair Encoding (BPE)

Implement a simple **BPE tokenizer** — the foundation of GPT/LLaMA tokenization.

### Signature
```python
class SimpleBPE:
    def __init__(self): ...
    def train(self, corpus: list[str], num_merges: int): ...
    def encode(self, text: str) -> list[str]: ...
```

### Algorithm (training)
1. Split each word into characters + `</w>` end marker
2. Count all adjacent pairs across the corpus
3. Merge the most frequent pair into a single token
4. Repeat for `num_merges` iterations

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.7 MB/s eta 0:00:00


In [ ]:
# No imports needed

In [25]:
# ✏️ YOUR IMPLEMENTATION HERE

EOW = '</w>'

class SimpleBPE:
    def __init__(self):
        self.merges = []

    def train(self, corpus, num_merges):
      # iteratively find & merge most frequent pairs
      vocab = {}
      for word in corpus:
        symbols = tuple(word) + (EOW,)
        vocab[symbols] = vocab.get(symbols, 0) + 1

      self.merges = []
      for _ in range(num_merges):
        pairs = {}
        for word, freq in vocab.items():
          for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            pairs[pair] = pairs.get(pair, 0) + 1
        if not pairs:
          break

        best = max(pairs, key=pairs.get)


        new_vocab = {}
        for word, freq in vocab.items():
          new_word = []
          i = 0
          while i < len(word):
            if i < len(word) - 1 and (word[i], word[i+1]) == best:
              new_word.append(word[i] + word[i+1])
              i = i + 2
            else:
              new_word.append(word[i])
              i = i + 1
          new_vocab[tuple(new_word)] = freq
        self.merges.append(best)
        vocab = new_vocab



    def encode(self, text):
      # apply learned merges to split text
      all_tokens = []
      for word in text.split():
        symbols = list(word) + [EOW]
        print(symbols)
        for a, b in self.merges:
          i = 0
          while i < len(symbols) - 1:
            if symbols[i] == a and symbols[i+ 1] == b:
              symbols = symbols[:i] + [a + b] + symbols[i+2:]
              print(symbols)
            else:
              i += 1

        all_tokens.extend(symbols)
      return all_tokens



In [26]:
# 🧪 Debug
bpe = SimpleBPE()
bpe.train(['low', 'low', 'low', 'lower', 'newest', 'widest'], num_merges=10)
print('Merges:', bpe.merges[:5])
print('Encode:', bpe.encode('low lower newest'))

Merges: [('l', 'o'), ('lo', 'w'), ('e', 's'), ('es', 't'), ('est', '</w>')]
['l', 'o', 'w', '</w>']
['lo', 'w', '</w>']
['low', '</w>']
['low</w>']
['l', 'o', 'w', 'e', 'r', '</w>']
['lo', 'w', 'e', 'r', '</w>']
['low', 'e', 'r', '</w>']
['lowe', 'r', '</w>']
['lower', '</w>']
['lower</w>']
['n', 'e', 'w', 'e', 's', 't', '</w>']
['n', 'e', 'w', 'es', 't', '</w>']
['n', 'e', 'w', 'est', '</w>']
['n', 'e', 'w', 'est</w>']
['ne', 'w', 'est</w>']
Encode: ['low</w>', 'lower</w>', 'ne', 'w', 'est</w>']


In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('bpe')